In [52]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm
from datetime import datetime
from functools import reduce

In [53]:
# --- 1단계: 데이터 준비 및 전처리 ---
print("1. 데이터 준비 및 전처리 시작...")


1. 데이터 준비 및 전처리 시작...


In [54]:
# 경기종합지수
ECONOMIC_INDEX = '../../data/raw/경기종합지수.csv'
# 공종별 건설기성액
CONSTRUCTION_WORK = '../../data/raw/공종별_건설기성액.csv'
# 부동산시장 소비심리지수
REAL_ESTATE_CONSUMER_INDEX = '../../data/raw/부동산시장_소비심리지수.csv'
# 소비자 물가지수
CPI_PATH = '../../data/raw/소비자_물가지수.csv'
# 주택시장 소비심리지수
HOUSING_CONSUMER_SENTIMENT = '../../data/raw/주택시장_소비심리지수.csv'
# 지역별 지가변동률
PRICE_CHANGE_RATE = '../../data/raw/지역별_지가변동률.csv'
# 토지시장 소비심리지수
LAND_MARKET_CONSUMER_SENTIMENT_INDEX = '../../data/raw/토지시장_소비심리지수.csv'
# 행정구역별_아파트매매거래현황
APT_TRANSACTIONS = '../../data/raw/행정구역별_아파트매매거래현황.csv'
# 금리데이터
INTEREST_RATE = '../../data/raw/201007-202507.csv'

# 뉴스 데이터
KINDS_PATH = '../../data/interim/news/deep_search_news.csv'
# 불용어
STOPWORD_PATH = '../../data/raw/news/stopwords-ko.txt'
# 긍정, 부정, 중립
POSITIVE_PATH = '../../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../../data/raw/news/sentiment/natural.txt'

In [55]:
#경기종합지수
EI = pd.read_csv(ECONOMIC_INDEX, encoding='cp949')
#공종별 건설기성액
CW = pd.read_csv(CONSTRUCTION_WORK, encoding='cp949')
#부동산시장 소비심리지수
RECI = pd.read_csv(REAL_ESTATE_CONSUMER_INDEX, encoding='cp949')
#소비자 물가지수
CPI = pd.read_csv(CPI_PATH, encoding='cp949')
#주택시장 소비심리지수
HCS = pd.read_csv(HOUSING_CONSUMER_SENTIMENT, encoding='cp949')
#지역별 지가변동률
PCR = pd.read_csv(PRICE_CHANGE_RATE, encoding='cp949')
#토지시장 소비심리지수
LMCSI = pd.read_csv(LAND_MARKET_CONSUMER_SENTIMENT_INDEX, encoding='cp949')
#행정구역별 아파트 매매거래현황
AT = pd.read_csv(APT_TRANSACTIONS, encoding='cp949')
# 금리
IR = pd.read_csv(INTEREST_RATE)
IR = IR.iloc[:,13:]

# 뉴스

In [56]:
# 뉴스 데이터 읽기
df = pd.read_csv(KINDS_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")
print(f"수집된 기사의 길이 : {len(df)}")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...
수집된 기사의 길이 : 37437


In [57]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    if not isinstance(text, str):
        return []
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['오후', '대전시', '지족동', '아파트', '지하', '주차장', '주차', '승용차', '차량', '아파트', '경비원', '진화']


In [58]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [59]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...   
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...   
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...   
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...   
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...   

                                   textrank_keywords  
0  [아파트, 대전시, 경비원, 지족동, 주차, 주차장, 승용차, 지하, 차량, 오후,...  
1  [중공업, 지난달, 동구, 현대, 평균, 울산시, 기록, 본격, 중단, 발표, 가동...  
2  [사업, 조성, 부지, 거제, 고현항, 개발, 녹지, 주거, 공원, 문화, 부족, ...  
3  [매매, 산리, 전원, 삼익, 비치, 생면, 주택, 남천동, 욕실, 주거, 지역, ...  
4  [주택, 시장, 부담, 종부, 인상, 반사, 건물, 상업, 소형, 관망세, 작용, ...  


In [60]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...         0.000000
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...        -0.075000
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...         0.090909
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...         0.240000
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...        -0.187500


In [61]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [62]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [63]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.strftime("%Y-%m")

In [64]:
print("\n월별 감성 지수:")
print(monthly_sentiment)
monthly_sentiment


월별 감성 지수:
      month  sentiment_score
0   2018-07         0.016983
1   2018-08         0.035666
2   2018-09         0.013844
3   2018-10         0.023940
4   2018-11         0.004340
..      ...              ...
82  2025-05         0.045637
83  2025-06         0.003236
84  2025-07        -0.007471
85  2025-08         0.010208
86  2025-09         0.007458

[87 rows x 2 columns]


,month,sentiment_score
0,2018-07,0.016983
1,2018-08,0.035666
2,2018-09,0.013844
3,2018-10,0.023940
4,2018-11,0.004340
...,...,...
82,2025-05,0.045637
83,2025-06,0.003236
84,2025-07,-0.007471
85,2025-08,0.010208


# 경기종합지수

In [65]:
EI = EI.iloc[2:,1:]

# 자료시점 컬럼을 datetime으로 변환
EI["자료시점"] = pd.to_datetime(EI["자료시점"], format="%Y년 %m월")
# "YYYY-MM" 형식으로 변환
EI["자료시점"] = EI["자료시점"].dt.strftime("%Y-%m")

EI["선행종합지수"]=EI['선행종합지수'].astype(float)
EI = EI.reset_index(drop=True)

EI = EI.rename(columns={"자료시점":"month","선행종합지수":"leading_index"})


In [66]:
EI.info()
EI.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   month          84 non-null     object 
 1   leading_index  84 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB


,month,leading_index
0,2018-07,94.7
1,2018-08,94.6


# 지역별 지가변동률

In [67]:
PCR.rename(columns={"자료시점":"month","서울":"강남구_변동률","서울.1":"강남구_누계"}, inplace=True)

PCR = PCR.iloc[3:, 1:]
PCR['month'] = pd.to_datetime(PCR['month'], format="%Y년 %m월")
PCR['month'] = PCR['month'].dt.strftime("%Y-%m")
PCR[['강남구_변동률','강남구_누계']] = PCR[['강남구_변동률','강남구_누계']].astype(float)

PCR.info()
PCR.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 3 to 86
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   month    84 non-null     object 
 1   강남구_변동률  84 non-null     float64
 2   강남구_누계   84 non-null     float64
dtypes: float64(2), object(1)
memory usage: 2.1+ KB


,month,강남구_변동률,강남구_누계
3,2018-07,0.692,2.975
4,2018-08,0.777,3.775


# 부동산시장 소비심리지수

In [68]:
RECI.rename(columns={"자료시점":"month","수도권":"부동산_소비심리지수"},inplace=True)

RECI = RECI.iloc[3:,1:]
RECI['month'] = pd.to_datetime(RECI['month'], format='%Y년 %m월')
RECI['month'] = RECI['month'].dt.strftime("%Y-%m")
RECI['부동산_소비심리지수'] = RECI['부동산_소비심리지수'].astype(float)

RECI.info()
RECI.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 3 to 86
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   month       84 non-null     object 
 1   부동산_소비심리지수  84 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB


,month,부동산_소비심리지수
3,2018-07,111.9
4,2018-08,127.5


# 주택시장 소비심리지수

In [69]:
HCS.rename(columns={"자료시점":"month","수도권":"주택시장_소비심리지수"},inplace=True)

HCS = HCS.iloc[3:, 1:]
HCS['month'] = pd.to_datetime(HCS['month'], format="%Y년 %m월")
HCS['month'] = HCS['month'].dt.strftime("%Y-%m")
HCS["주택시장_소비심리지수"] = HCS['주택시장_소비심리지수'].astype(float)

HCS.info()
HCS.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 3 to 86
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   month        84 non-null     object 
 1   주택시장_소비심리지수  84 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB


,month,주택시장_소비심리지수
3,2018-07,113.4
4,2018-08,129.7


# 토지시장 소비심리지수

In [70]:
LMCSI.rename(columns={'자료시점':'month','수도권':'토지시장_소비심리지수'},inplace=True)

In [71]:
LMCSI = LMCSI.iloc[3:,1:]

In [72]:
LMCSI['month'] = pd.to_datetime(LMCSI['month'], format="%Y년 %m월")

In [73]:
LMCSI['month'] = LMCSI['month'].dt.strftime("%Y-%m")

In [74]:
LMCSI['토지시장_소비심리지수'] = LMCSI['토지시장_소비심리지수'].astype(float)

In [75]:
LMCSI.info()
LMCSI.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 3 to 86
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   month        84 non-null     object 
 1   토지시장_소비심리지수  84 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB


,month,토지시장_소비심리지수
3,2018-07,98.3
4,2018-08,108.1


# 아파트 매매거래 현황

In [76]:
AT.rename(columns={'자료시점':'month','서울':'아파트_호수','서울.1':'아파트_면적'},inplace=True)

In [77]:
AT = AT.iloc[3:,1:]

In [78]:
AT['month'] = pd.to_datetime(AT['month'], format="%Y년 %m월")

In [79]:
AT['month'] = AT['month'].dt.strftime("%Y-%m")

In [80]:
AT[['아파트_호수','아파트_면적']] = AT[['아파트_호수','아파트_면적']].astype(float)

In [81]:
AT.info()
AT.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 3 to 86
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   month   84 non-null     object 
 1   아파트_호수  84 non-null     float64
 2   아파트_면적  84 non-null     float64
dtypes: float64(2), object(1)
memory usage: 2.1+ KB


,month,아파트_호수,아파트_면적
3,2018-07,183.0,18.0
4,2018-08,256.0,22.0


# 공종별 건설기성액

In [82]:
CW.rename(columns={"자료시점":"month",'계':"건설기성액(백만원)"},inplace=True)

In [83]:
CW = CW.iloc[2:,1:]

In [84]:
CW['month'] = pd.to_datetime(CW['month'], format="%Y년 %m월")

In [85]:
CW['month'] = CW['month'].dt.strftime("%Y-%m")

In [86]:
CW['건설기성액(백만원)'] = CW['건설기성액(백만원)']\
    .str.replace('"', '')\
    .str.replace(',', '')\
    .astype(float)

In [87]:
CW.info()
CW.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 2 to 85
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   month       84 non-null     object 
 1   건설기성액(백만원)  84 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.4+ KB


,month,건설기성액(백만원)
2,2018-07,11215836.0
3,2018-08,11146209.0


# 금리

In [88]:
IR = IR.melt(var_name='date', value_name='rate')
# datetime 변환
IR.rename(columns={"date":"month"}, inplace=True)
IR['month'] = pd.to_datetime(IR['month'], format='%Y/%m')
IR['month'] = IR['month'].dt.strftime('%Y-%m')

In [89]:
IR.info()
IR.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   month   180 non-null    object 
 1   rate    180 non-null    float64
dtypes: float64(1), object(1)
memory usage: 2.9+ KB


,month,rate
0,2010-07,2.25
1,2010-08,2.25


# 모든 데이터 병합

In [102]:
SALE_PATH = '../../data/interim/apt/gang_nam_apt_with_long_lat.csv'
sale = pd.read_csv(SALE_PATH)
len(sale)

21561

In [103]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.strftime("%Y-%m")
drop_col = ['단지명','도로명','경도','위도']

In [104]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '계약일자',
       '경도', '위도', 'month'],
      dtype='object')

In [105]:
sale.drop(drop_col, axis=1, inplace=True)

In [106]:
sale.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,계약일자,month
0,84.73,5,1994,7.399156,24,2018-07-01,2018-07
1,169.19,4,2003,6.753467,15,2018-07-01,2018-07
2,84.43,10,1980,7.618163,38,2018-07-02,2018-07
3,76.79,4,1979,7.570627,39,2018-07-02,2018-07
4,162.51,1,1999,6.898420,19,2018-07-02,2018-07


In [107]:
dfs = [sale, monthly_sentiment, EI, CW, RECI, HCS, PCR, LMCSI, AT, IR]

In [108]:
for df in dfs:
    print(df.columns)

Index(['전용면적(㎡)', '층', '건축년도', '면적당 단가(만원)', '아파트 나이', '계약일자', 'month'], dtype='object')
Index(['month', 'sentiment_score'], dtype='object')
Index(['month', 'leading_index'], dtype='object')
Index(['month', '건설기성액(백만원)'], dtype='object')
Index(['month', '부동산_소비심리지수'], dtype='object')
Index(['month', '주택시장_소비심리지수'], dtype='object')
Index(['month', '강남구_변동률', '강남구_누계'], dtype='object')
Index(['month', '토지시장_소비심리지수'], dtype='object')
Index(['month', '아파트_호수', '아파트_면적'], dtype='object')
Index(['month', 'rate'], dtype='object')


In [109]:
merged_df = reduce(lambda left, right: pd.merge(left, right, on='month', how='left'), dfs)

In [110]:
merged_df.drop('month', axis=1, inplace=True)

In [111]:
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,계약일자,sentiment_score,leading_index,건설기성액(백만원),부동산_소비심리지수,주택시장_소비심리지수,강남구_변동률,강남구_누계,토지시장_소비심리지수,아파트_호수,아파트_면적,rate
0,84.73,5,1994,7.399156,24,2018-07-01,0.016983,94.7,11215836.0,111.9,113.4,0.692,2.975,98.3,183.0,18.0,1.5
1,169.19,4,2003,6.753467,15,2018-07-01,0.016983,94.7,11215836.0,111.9,113.4,0.692,2.975,98.3,183.0,18.0,1.5
2,84.43,10,1980,7.618163,38,2018-07-02,0.016983,94.7,11215836.0,111.9,113.4,0.692,2.975,98.3,183.0,18.0,1.5
3,76.79,4,1979,7.570627,39,2018-07-02,0.016983,94.7,11215836.0,111.9,113.4,0.692,2.975,98.3,183.0,18.0,1.5
4,162.51,1,1999,6.898420,19,2018-07-02,0.016983,94.7,11215836.0,111.9,113.4,0.692,2.975,98.3,183.0,18.0,1.5


In [112]:
len(merged_df)

21561

In [113]:
merged_df.to_csv('../../data/interim/gang_nam_sendimental_score_with_sale.csv',index=False)